# Report Agent Evaluation

In this notebook, we'll evaluate the Report Generation Agent from Module 1. You'll learn how to assess report quality using LLM-as-a-judge techniques with NVIDIA models to evaluate structure, content, accuracy, and writing quality.

## Setup

First, let's install dependencies and load environment variables.

In [ ]:
%pip install -r ../../requirements.txt > /dev/null
from dotenv import load_dotenv
_ = load_dotenv("../../variables.env")
_ = load_dotenv("../../secrets.env")

## Import Libraries

In [ ]:
import json
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any

from evaluation_framework import (
    create_judge_llm,
    evaluate_report_quality,
)

print("Libraries imported successfully!")

## Load Test Cases

We've prepared test cases with diverse report topics. Each includes:
- **Topic**: The subject for the report
- **Expected Sections**: Sections that should appear in a good report
- **Quality Criteria**: Standards for evaluating the report

In [ ]:
# Load test cases
report_test_cases_path = Path("../data/evaluation/report_agent_test_cases.json")
with open(report_test_cases_path, 'r') as f:
    report_test_cases = json.load(f)

print(f"Loaded {len(report_test_cases)} test cases\n")
print("Example test case:")
print(f"Topic: {report_test_cases[0]['topic']}")
print(f"Expected sections: {', '.join(report_test_cases[0]['expected_sections'][:3])}...")

## Initialize the Report Generation Agent

Let's load the Report Generation Agent from Module 1.

In [ ]:
# Import the document generation agent
from docgen_agent import agent

print("Report generation agent loaded successfully!")
print("Agent is ready to generate research reports.")

## Generate Reports

Now let's generate reports for each test topic.

**Note**: This will take significant time (5-10 minutes per report) as each report requires:
1. Initial topic research
2. Report planning and outlining
3. Section-by-section authoring with additional research
4. Final compilation

In [ ]:
results = []

print("Generating reports...\n")
print("=" * 80)
print("This may take 30-60 minutes total. Please be patient.\n")

for i, test_case in enumerate(report_test_cases):
    print(f"\n[{i+1}/{len(report_test_cases)}] Generating report on: {test_case['topic']}")
    print(f"Expected sections: {len(test_case['expected_sections'])}")
    
    try:
        # Generate report
        result = await agent.graph.ainvoke({
            "topic": test_case["topic"],
            "report_structure": "standard",
            "messages": []
        })
        
        report_content = result.get("report", "")
        
        results.append({
            "topic": test_case["topic"],
            "report": report_content,
            "expected_sections": test_case["expected_sections"],
            "quality_criteria": test_case.get("quality_criteria", {})
        })
        
        print(f"Report generated: {len(report_content)} characters")
        print(f"Preview: {report_content[:150]}...")
        
    except Exception as e:
        print(f"Error: {str(e)}")
        results.append({
            "topic": test_case["topic"],
            "report": f"ERROR: {str(e)}",
            "expected_sections": test_case["expected_sections"],
            "quality_criteria": test_case.get("quality_criteria", {})
        })

print("\n" + "=" * 80)
print(f"\nCompleted generating {len(results)} reports!")

## Evaluate Report Quality

Now let's evaluate each report using NVIDIA Nemotron as a judge. We'll assess:

1. **Structure**: Are all expected sections present and well-organized?
2. **Content**: Is the information substantive and relevant?
3. **Accuracy**: Are claims well-supported and factual?
4. **Writing**: Is it clear, professional, and well-written?

In [ ]:
# Initialize judge model
print("Initializing NVIDIA Nemotron judge model...")
judge_llm = create_judge_llm()
print("Judge model ready!\n")

print("Evaluating reports...")
print("=" * 80)

# Evaluate each report
for i, result in enumerate(results):
    print(f"\n[{i+1}/{len(results)}] Evaluating: {result['topic'][:60]}...")
    
    try:
        # Run report quality evaluation
        eval_results = evaluate_report_quality(
            topic=result["topic"],
            report=result["report"],
            expected_sections=result["expected_sections"],
            judge_llm=judge_llm
        )
        
        # Store evaluation results (normalized to 0-1 scale)
        result["structure_score"] = eval_results["structure"].score / 5.0
        result["structure_explanation"] = eval_results["structure"].explanation
        result["content_score"] = eval_results["content"].score / 5.0
        result["content_explanation"] = eval_results["content"].explanation
        result["accuracy_score"] = eval_results["accuracy"].score / 5.0
        result["accuracy_explanation"] = eval_results["accuracy"].explanation
        result["writing_score"] = eval_results["writing"].score / 5.0
        result["writing_explanation"] = eval_results["writing"].explanation
        
        # Calculate aggregate
        result["aggregate_score"] = sum([
            result["structure_score"],
            result["content_score"],
            result["accuracy_score"],
            result["writing_score"]
        ]) / 4.0
        
        print(f"  Structure: {result['structure_score']:.2f} - {eval_results['structure'].explanation[:50]}...")
        print(f"  Content:   {result['content_score']:.2f} - {eval_results['content'].explanation[:50]}...")
        print(f"  Accuracy:  {result['accuracy_score']:.2f} - {eval_results['accuracy'].explanation[:50]}...")
        print(f"  Writing:   {result['writing_score']:.2f} - {eval_results['writing'].explanation[:50]}...")
        print(f"  Overall:   {result['aggregate_score']:.2f}")
        
    except Exception as e:
        print(f"Error during evaluation: {str(e)}")
        result["structure_score"] = 0.0
        result["content_score"] = 0.0
        result["accuracy_score"] = 0.0
        result["writing_score"] = 0.0
        result["aggregate_score"] = 0.0

print("\n" + "=" * 80)
print("\nEvaluation complete!")

## Analyze Results

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(results)

print("=" * 80)
print("OVERALL EVALUATION RESULTS")
print("=" * 80)
print(f"\nMean Scores (0-1 scale):")
print(f"  Structure:  {df['structure_score'].mean():.3f}")
print(f"  Content:    {df['content_score'].mean():.3f}")
print(f"  Accuracy:   {df['accuracy_score'].mean():.3f}")
print(f"  Writing:    {df['writing_score'].mean():.3f}")
print(f"\nAggregate Score: {df['aggregate_score'].mean():.3f}")

print("\n" + "=" * 80)
print("SCORE DISTRIBUTION")
print("=" * 80)
for metric in ['structure_score', 'content_score', 'accuracy_score', 'writing_score']:
    print(f"\n{metric.replace('_score', '').capitalize()}:")
    print(df[metric].describe())

## Identify Problem Areas

In [ ]:
threshold = 0.6

print("=" * 80)
print(f"LOW-SCORING REPORTS (Score < {threshold})")
print("=" * 80)

low_scores = df[
    (df['structure_score'] < threshold) |
    (df['content_score'] < threshold) |
    (df['accuracy_score'] < threshold) |
    (df['writing_score'] < threshold)
]

if len(low_scores) > 0:
    print(f"\nFound {len(low_scores)} reports with low scores:\n")
    
    for idx, row in low_scores.iterrows():
        print(f"\nTopic: {row['topic']}")
        print(f"Report length: {len(row['report'])} characters")
        print(f"\nScores:")
        print(f"  Structure: {row['structure_score']:.2f} - {row['structure_explanation']}")
        print(f"  Content:   {row['content_score']:.2f} - {row['content_explanation']}")
        print(f"  Accuracy:  {row['accuracy_score']:.2f} - {row['accuracy_explanation']}")
        print(f"  Writing:   {row['writing_score']:.2f} - {row['writing_explanation']}")
        print("-" * 80)
else:
    print("\nExcellent! All reports scored above threshold.")

## Report Length Analysis

In [ ]:
df['report_length'] = df['report'].apply(len)

print("=" * 80)
print("REPORT LENGTH ANALYSIS")
print("=" * 80)
print(f"\nMean length: {df['report_length'].mean():.0f} characters")
print(f"Min length:  {df['report_length'].min():.0f} characters")
print(f"Max length:  {df['report_length'].max():.0f} characters")

# Check for placeholder content
df['has_placeholder'] = df['report'].apply(
    lambda x: "will be drafted" in x.lower() or "to be written" in x.lower()
)
placeholder_count = df['has_placeholder'].sum()

if placeholder_count > 0:
    print(f"\nWarning: {placeholder_count} reports contain placeholder content")

## Save Results

In [ ]:
# Ensure output directory exists
output_dir = Path("../data/evaluation")
output_dir.mkdir(parents=True, exist_ok=True)

# Save detailed results
results_path = output_dir / "report_agent_results.csv"
df.to_csv(results_path, index=False)
print(f"Detailed results saved to: {results_path}")

# Save summary
summary = {
    "timestamp": pd.Timestamp.now().isoformat(),
    "total_test_cases": len(df),
    "mean_scores": {
        "structure": float(df['structure_score'].mean()),
        "content": float(df['content_score'].mean()),
        "accuracy": float(df['accuracy_score'].mean()),
        "writing": float(df['writing_score'].mean()),
        "aggregate": float(df['aggregate_score'].mean())
    },
    "mean_length": float(df['report_length'].mean()),
    "low_scoring_count": len(low_scores),
    "placeholder_count": int(placeholder_count)
}

summary_path = output_dir / "report_agent_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved to: {summary_path}")

print("\n" + "=" * 80)
print("RESULTS SAVED")
print("=" * 80)

## Improvement Recommendations

In [ ]:
print("=" * 80)
print("IMPROVEMENT RECOMMENDATIONS")
print("=" * 80)

recommendations = []

if df['structure_score'].mean() < 0.8:
    recommendations.append(
        "Structure: Improve report planning to ensure all expected sections are included."
    )

if df['content_score'].mean() < 0.8:
    recommendations.append(
        "Content: Enhance research depth for more substantive information."
    )

if df['accuracy_score'].mean() < 0.8:
    recommendations.append(
        "Accuracy: Strengthen fact-checking and source verification."
    )

if df['writing_score'].mean() < 0.8:
    recommendations.append(
        "Writing: Improve writing quality through better prompts."
    )

if placeholder_count > 0:
    recommendations.append(
        "Placeholders: Investigate why sections contain placeholder content."
    )

if df['report_length'].mean() < 2000:
    recommendations.append(
        "Length: Reports may be too short. Adjust prompts for more depth."
    )

if len(recommendations) == 0:
    print("\nExcellent! Report agent performing well.")
else:
    print("\nRecommendations:\n")
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec}")

print("\n" + "=" * 80)

## Next Steps

Congratulations! You've evaluated your Report Generation Agent.

1. **Implement Improvements**: Use recommendations to enhance the agent
2. **Re-evaluate**: Measure the impact of changes
3. **Add Test Cases**: Expand coverage with more topics
4. **Monitor Production**: Track performance over time

Continue to [Continuous Improvement](../.devx/3-agent-evaluation/continuous_improvement.md).